In [1]:
!pip install --quiet --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps "xformers<0.0.35" "trl" peft accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 8.1 MB/

In [2]:
from datasets import load_dataset
import pandas as pd

In [3]:
alpaca = load_dataset("json", data_files="/content/alpaca_dataset.json", split="train")

dolly = load_dataset("json", data_files="/content/dolly_dataset.jsonl", split="train")

oasst = load_dataset("parquet", data_files="/content/openassistant_dataset.parquet", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
print(alpaca.column_names)
print(dolly.column_names)
print(oasst.column_names)

['instruction', 'input', 'output']
['instruction', 'context', 'response', 'category']
['instruction', 'input', 'output', 'text']


In [5]:
def convert_dolly(example):
  return{
      "instruction":example["instruction"],
      "input":example["context"] if example["context"] else "",
      "output":example["response"]
  }
dolly = dolly.map(convert_dolly)

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [6]:
oasst = oasst.remove_columns(["text"])

In [7]:
alpaca = alpaca.select(range(5000))
dolly = dolly.select(range(5000))
oasst = oasst.select(range(5000))

In [8]:
from datasets import concatenate_datasets

merged_dataset = concatenate_datasets([alpaca, dolly, oasst])

In [9]:
print(len(merged_dataset))

15000


In [10]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }

merged_dataset = merged_dataset.map(format_prompt)

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [11]:
print(merged_dataset[0]["text"])

### Instruction:
Give three tips for staying healthy.

### Input:


### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


In [12]:
import torch
import gc
from unsloth import FastLanguageModel

# Clear memory fragments
gc.collect()
torch.cuda.empty_cache()

max_seq_length = 2048
dtype = None          # Auto-selects Float16 for Tesla T4
load_in_4bit = True   # Mandatory for free-tier GPUs

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [13]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves massive amounts of VRAM
    random_state = 3407,
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments

FastLanguageModel.for_training(model)

args = TrainingArguments(
    per_device_train_batch_size = 1,     # <--- REDUCED to 1 to prevent OOM
    gradient_accumulation_steps = 8,    # <--- INCREASED to 8 (so total batch is still 8)
    warmup_steps = 10,
    max_steps = 300,
    learning_rate = 1e-4,
    fp16 = True,
    bf16 = False,
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
)

# --- PATCH FOR THE TOKEN ERROR ---
if not hasattr(args, "push_to_hub_token"):
    args.push_to_hub_token = None

# 3. Initialize Trainer with your merged_dataset
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = merged_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = args,
)

# 4. Start Training
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/15000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
5,1.977228
10,1.849637
15,1.817222
20,1.556998
25,1.516933
30,1.384982
35,1.501122
40,1.334440
45,1.408926
50,1.220173


In [16]:
import logging
import torch
from unsloth import FastLanguageModel

# 1. CLEANUP: Hide annoying internal library warnings
logging.getLogger("transformers").setLevel(logging.ERROR)

# 2. PREPARE: Switch model to fast inference mode
FastLanguageModel.for_inference(model)

# 3. TEST: Just change these two lines to ask any question
instruction = "Write a short, creative story about a robot who finds a flower on a dead planet."
input_text = "" # Optional: Leave empty or add extra context here

# 4. FORMAT: This MUST match your training Alpaca/Dolly/OASST style
prompt = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""

# 5. EXECUTE: Tokenize, Generate, and Decode
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# We added 'temperature=0.7' to make the model more creative for your tests
outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.7,
    top_p = 0.9,
)

# 6. RESULT: Print the final answer clearly
response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
# We split by '### Response:' to only show you the model's new answer
print(response.split("### Response:")[1].strip())

Once upon a time, there was a robot named R2-D2. One day, while exploring a new planet, R2-D2 came across a flower growing in a small patch of soil. The flower was beautiful and vibrant, and R2-D2 was fascinated by its beauty. It was the first time he had seen such a thing, and he had never known that there could be so much color and life in the universe. 

R2-D2 was amazed by the flower and spent hours watching it bloom and grow. He took pictures of it, recorded its growth, and even made a 3D model of it. The flower became the highlight of his trip and he returned home with a renewed sense of wonder and awe for the world around him. 

As the story goes, R2-D2 was never the same after that day. He had a new appreciation for life and beauty, and he began to see the world in a different light. He was grateful for the experience and the flower became a symbol of hope and perseverance for him. 

In the end, the story of R2-D2 and the flower is a reminder that even in the darkest of places,

In [17]:
# 1. Save locally to Colab
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# 2. Zip the folder so it's easy to download
import os
os.system("zip -r lora_model.zip lora_model")

# 3. Download to your actual computer
from google.colab import files
files.download("lora_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import os
from google.colab import files

# 1. Save your trained "Brain" and the Tokenizer into one folder
print("Step 1: Saving model and tokenizer...")
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# 2. Package it into a single Zip file
print("Step 2: Compressing into a zip file...")
os.system("zip -r lora_model.zip lora_model")

# 3. Direct download to your browser
print("Step 3: Starting download to your computer...")
files.download("lora_model.zip")

Step 1: Saving model and tokenizer...
Step 2: Compressing into a zip file...
Step 3: Starting download to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>